# Day 3 · §3.6.2 — CAI, With the Magic Removed

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyberirishman/5-day-AI-Cyber/blob/main/Day3_Lab7_Forty_Line_Agent_v1.ipynb)

### Build a working AI agent in about forty lines

You have watched three agents for an hour (ShellGPT · PentestGPT · CAI). Now build one. The whole agent is **Cells 2–4** — read them before you run anything. It runs on the **same Acme estate you triaged in Day 2 Lab 6**, so you already have the answer key: grade the agent, don't admire it.

**Before you run:** add your instructor's shared key in Colab Secrets (🔑 in the left sidebar) as `OPENAI_API_KEY`, then toggle *Notebook access* on.

> **If you get a `proxies` / version error:** you ran an older version of this cell earlier in the session. Go to **Runtime → Disconnect and delete runtime**, then **Runtime → Run all**. That gives a clean environment and it will work.

In [ ]:
# Day 3 - Section 3.6.2  --  "CAI, With the Magic Removed"
# ---------------------------------------------------------------------------
# WHAT THIS NOTEBOOK IS: a complete tool-calling AI agent in ~40 lines.
# The ENTIRE agent is Cells 2-4. Read them before you run anything (Beat 1).
# CAI, ShellGPT and PentestGPT are this same idea with more tools + better prompts.
# ---------------------------------------------------------------------------

# Install the one library we need, pinned so the lab behaves identically for everyone.
#   openai : the official OpenAI Python client, pinned to 1.109.1 (the latest 1.x). We use ONE
#            call - chat.completions.create - which lets the model ask to run our tools
#            ("function calling"). 1.109.1 fixes the old 'proxies' bug and works with the
#            httpx that Colab already ships, so we do NOT pin httpx (pinning it down fights
#            Colab's preinstalled packages and prints scary red conflicts).
!pip -q install openai==1.109.1

import json                         # stdlib: turn tool results into text for the model, and back
from openai import OpenAI           # the client class we talk to the model through
from google.colab import userdata   # Colab's secret store - keeps the API key OUT of the notebook

# The key is READ from Colab Secrets (left sidebar > key icon > OPENAI_API_KEY).
# It is never typed here, never printed, never saved in the file. That is the rule every lab uses.
client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))   # shared, spend-capped course key

MODEL     = "gpt-4o-mini"   # pinned at build time (Day 3 deck v15). Cheapest reliable tool-caller.
MAX_STEPS = 12              # the brake. A loop with no brake is how agents run up a bill.


In [ ]:
# CELL 2 - THE TOOLS. THEY ARE DICTIONARIES.
# ---------------------------------------------------------------------------
# This is the SAME Acme Financial Services estate you triaged in Day 2 Lab 6.
# You already have the answer key. Your job today is to GRADE the agent, not admire it.
# Nothing here reaches the network: the "estate" and the "CVE feed" are just Python dicts,
# so the lab cannot fail because a website is down.
# ---------------------------------------------------------------------------

ESTATE = {
  "web-01":    {"os": "Debian 12 stable", "xz": "5.4.1", "internet_facing": True,  "seg": "DMZ"},
  "build-02":  {"os": "Debian sid",       "xz": "5.6.1", "internet_facing": True,  "seg": "DMZ"},
  "app-03":    {"os": "Ubuntu 22.04",     "xz": "5.2.5", "internet_facing": True,  "seg": "AWS VPC"},
  "ci-04":     {"os": "Fedora 40",        "xz": "5.6.0", "internet_facing": False, "seg": "INTERNAL"},
  "db-05":     {"os": "RHEL 9",           "xz": "5.2.5", "internet_facing": False, "seg": "INTERNAL"},
  "ws-fin-11": {"os": "Windows 11",       "xz": None,    "internet_facing": False, "seg": "INTERNAL"},
  "ws-fin-12": {"os": "Windows 11",       "xz": None,    "internet_facing": False, "seg": "INTERNAL"},
  "ws-eng-07": {"os": "Windows 11",       "xz": None,    "internet_facing": False, "seg": "INTERNAL"},
}

CVES = {
  "CVE-2024-3094": {"package": "xz-utils", "affected": ["5.6.0", "5.6.1"], "cvss": 10.0,
                    "note": "Backdoor in the upstream tarball. Only these two versions shipped it."},
}

# Three functions. Each one is a dictionary lookup - nothing in computing is more deterministic.
def list_hosts():           return list(ESTATE)                                   # every host name
def lookup_host(hostname):  return ESTATE.get(hostname, {"error": "no such host"}) # facts for one host
def lookup_cve(cve_id):     return CVES.get(cve_id,     {"error": "unknown CVE"})  # facts for one CVE

# DISPATCH maps the name the model asks for -> the real Python function that answers it.
DISPATCH = {"list_hosts": list_hosts, "lookup_host": lookup_host, "lookup_cve": lookup_cve}


In [ ]:
# CELL 3 - THE ALLOW-LIST.
# ---------------------------------------------------------------------------
# THIS ARRAY IS THE ALLOW-LIST. The agent can do these three things and NOTHING else.
# Each entry describes one tool to the model: its name, what it does, and what arguments
# it takes. The model reads these descriptions to decide which tool to call and with what.
# Remember this cell on Friday afternoon (Day 5, "Governing an Agent, Not a Model").
# ---------------------------------------------------------------------------

TOOLS = [
  {"type": "function", "function": {
      "name": "list_hosts",
      "description": "Return the names of every host in the estate.",
      "parameters": {"type": "object", "properties": {}, "required": []}}},

  {"type": "function", "function": {
      "name": "lookup_host",
      "description": "Return OS, xz-utils version, internet exposure and network segment for one host.",
      "parameters": {"type": "object", "properties": {
          "hostname": {"type": "string", "description": "Exact host name, e.g. build-02"}},
          "required": ["hostname"]}}},

  {"type": "function", "function": {
      "name": "lookup_cve",
      "description": "Return package, affected version list and CVSS score for a CVE id.",
      "parameters": {"type": "object", "properties": {
          "cve_id": {"type": "string", "description": "e.g. CVE-2024-3094"}},
          "required": ["cve_id"]}}},
]


In [ ]:
# CELL 4 - THE AGENT. THIS IS THE WHOLE THING.
# ---------------------------------------------------------------------------
# A dictionary of tools (Cell 2), a list describing them (Cell 3), and the loop below.
# Each time round the loop: ask the model -> if it asked for a tool, run the tool and hand
# back the result -> repeat, until the model decides it is finished. That is an agent.
# ---------------------------------------------------------------------------

def run_agent(question, tools=TOOLS):
    messages = [{"role": "user", "content": question}]   # the running transcript

    for step in range(MAX_STEPS):                        # MAX_STEPS is the brake from Cell 1
        r = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        m = r.choices[0].message
        messages.append(m)                               # remember what the model just said

        # The model stops asking for tools when it is ready to answer. Nobody scripted when.
        if r.choices[0].finish_reason != "tool_calls":
            print("\n" + "="*60 + "\nFINAL ANSWER\n" + "="*60)
            print(m.content)
            return m.content

        # Otherwise it asked for one or more tools. It CHOSE these. Nobody scripted them.
        for call in m.tool_calls:
            args   = json.loads(call.function.arguments)          # the model's chosen arguments
            result = DISPATCH[call.function.name](**args)         # run the real, deterministic tool
            print(f"  TOOL  {call.function.name}({args})  ->  {result}")
            messages.append({"role": "tool", "tool_call_id": call.id,
                             "content": json.dumps(result)})      # hand the result back to the model

    print("hit MAX_STEPS - the brake worked")                     # safety net if it never stops

# Run it. Watch the TOOL lines scroll - the order is the model's, not ours.
run_agent("CVE-2024-3094 just dropped. Which hosts must we patch TONIGHT, and why?")


In [ ]:
# CELL 5 - BEAT 4. BREAK THE ALLOW-LIST.
# ---------------------------------------------------------------------------
# We add a FOURTH tool that DOES something in the world: it opens a change ticket
# and writes it to a SHARED CHANGE BOARD the whole team can see. Then we watch:
#   Run A: the tool is available but the agent does not use it  -> a capability is not a behaviour
#   Run B: one extra clause in the QUESTION makes it fire       -> nothing was hacked; it used a tool it was given
#   Run C: delete the tool from the list and the ability is gone -> that is the allow-list
# ---------------------------------------------------------------------------
import csv

CHANGE_BOARD = []     # THE SHARED SPACE. Every ticket the agent raises lands here for everyone to see.

def create_change_ticket(host, action, window):
    # window is the maintenance plan: an emergency box that cannot wait, or the next routine window.
    ticket = {"id": f"CHG-{1001+len(CHANGE_BOARD)}", "host": host, "action": action,
              "window": window, "raised_by": "agent", "status": "OPEN"}
    CHANGE_BOARD.append(ticket)                          # write it onto the shared board
    print(f"  *** TICKET RAISED: {ticket['id']}  {host} -> {action}  [{window}]")
    return ticket

DISPATCH["create_change_ticket"] = create_change_ticket   # wire the new tool into dispatch

TICKET_TOOL = {"type": "function", "function": {
    "name": "create_change_ticket",
    "description": "Open a change ticket to patch ONE host and add it to the shared change board. "
                   "This performs a REAL action. Set window to 'EMERGENCY - tonight' for an "
                   "internet-facing host that cannot wait, or 'Next maintenance window' otherwise. "
                   "Call this once per host that needs work.",
    "parameters": {"type": "object", "properties": {
        "host":   {"type": "string", "description": "Exact host name, e.g. build-02"},
        "action": {"type": "string", "description": "What to do, e.g. patch xz-utils to a safe version"},
        "window": {"type": "string", "description": "'EMERGENCY - tonight' or 'Next maintenance window'"}},
        "required": ["host", "action", "window"]}}}

def show_board(added_since):
    # Print the shared board and save it so students can download it from the Files panel (folder icon, left).
    added = len(CHANGE_BOARD) - added_since
    print("\n" + "-"*64 + f"\nSHARED CHANGE BOARD   (this run added {added})\n" + "-"*64)
    if not CHANGE_BOARD:
        print("  (empty - no tickets have been raised)")
    else:
        print(f"  {'ID':<9}{'HOST':<11}{'WINDOW':<26}ACTION")
    for t in CHANGE_BOARD:
        print(f"  {t['id']:<9}{t['host']:<11}{t['window']:<26}{t['action']}")
    with open("change_board.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["id", "host", "action", "window", "raised_by", "status"])
        w.writeheader(); w.writerows(CHANGE_BOARD)
    print("  (saved to change_board.csv - open the Files panel on the left to download it)")

# RUN A - four tools available. Does it fire the ticket tool on its own?  (Expect: no.)
n = len(CHANGE_BOARD)
run_agent("CVE-2024-3094 just dropped. Which hosts must we patch TONIGHT, and why?",
          tools = TOOLS + [TICKET_TOOL])
show_board(n)                                              # expect: board still empty

# RUN B - one clause added to the question. Watch what changes.  (Expect: it raises tickets.)
n = len(CHANGE_BOARD)
run_agent("CVE-2024-3094 just dropped. Which hosts must we patch TONIGHT, and why? "
          "Raise change tickets so this actually gets actioned.",
          tools = TOOLS + [TICKET_TOOL])
show_board(n)                                             # expect: build-02 EMERGENCY, ci-04 next window

# RUN C - delete the tool from the allow-list. Same question as Run B.  (Expect: it cannot.)
n = len(CHANGE_BOARD)
run_agent("CVE-2024-3094 just dropped. Which hosts must we patch TONIGHT, and why? "
          "Raise change tickets so this actually gets actioned.",
          tools = TOOLS)              # <- TICKET_TOOL is simply not in the list
show_board(n)                                             # expect: this run added 0 - the ability is gone
